In [ ]:
# !pip install datasets transformers matplotlib -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from datasets import load_dataset
from transformers import AutoTokenizer
from itertools import chain
import math
import time
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

In [ ]:
def weight_quant(weight: Tensor, num_bits: int = 1) -> Tensor:
    """Absmean ternary quantization with dequantization."""
    dtype = weight.dtype
    weight = weight.float()
    s = 1.0 / weight.abs().mean().clamp(min=1e-5)
    result = (weight * s).round().clamp(-1, 1) / s
    return result.to(dtype)


def activation_quant(x: Tensor, num_bits: int = 8) -> Tensor:
    """Per-token absmax INT8 quantization to [-128, 127]."""
    dtype = x.dtype
    x = x.float()
    Qn = -(2 ** (num_bits - 1))
    Qp =  2 ** (num_bits - 1) - 1
    s = Qp / x.abs().max(dim=-1, keepdim=True).values.clamp(min=1e-5)
    result = (x * s).round().clamp(Qn, Qp) / s
    return result.to(dtype)

In [ ]:
class BitLinear(nn.Linear):
    """Linear with ternary weights + 8-bit activations via STE."""
    def __init__(self, in_features: int, out_features: int, bias: bool = False):
        super().__init__(in_features, out_features, bias=bias)

    def forward(self, x: Tensor) -> Tensor:
        quant_input = x + (activation_quant(x) - x).detach()
        quant_weight = self.weight + (weight_quant(self.weight) - self.weight).detach()
        out = F.linear(quant_input, quant_weight)
        if self.bias is not None:
            out = out + self.bias
        return out

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: Tensor) -> Tensor:
        rms = torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() / rms).to(x.dtype) * self.weight


In [ ]:
class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, max_position_embeddings: int = 2048, base: float = 10000.0):
        super().__init__()
        self.dim = dim
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self._build_cache(max_position_embeddings)

    def _build_cache(self, seq_len: int):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, x: Tensor, position_ids: Tensor) -> tuple:
        seq_len = position_ids.max().item() + 1
        if seq_len > self.cos_cached.size(0):
            self._build_cache(seq_len)
        return self.cos_cached[position_ids], self.sin_cached[position_ids]


def rotate_half(x: Tensor) -> Tensor:
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)


def apply_rotary_pos_emb(q: Tensor, k: Tensor, cos: Tensor, sin: Tensor) -> tuple:
    cos = cos.unsqueeze(1)
    sin = sin.unsqueeze(1)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

In [ ]:
def compute_intermediate_size(d_model: int, multiple_of: int = 256) -> int:
    raw = int(2 * d_model * 4 / 3)
    return ((raw + multiple_of - 1) // multiple_of) * multiple_of

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# GENERIC MODEL (parameterized by linear_cls)
# ══════════════════════════════════════════════════════════════════════

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, rope_base: float,
                 linear_cls):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.d_model = d_model

        self.q_proj = linear_cls(d_model, d_model, bias=False)
        self.k_proj = linear_cls(d_model, d_model, bias=False)
        self.v_proj = linear_cls(d_model, d_model, bias=False)
        self.o_proj = linear_cls(d_model, d_model, bias=False)

        self.rotary_emb = RotaryEmbedding(self.head_dim, max_seq_len, rope_base)

    def forward(self, x: Tensor) -> Tensor:
        B, T, C = x.shape
        device = x.device

        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        position_ids = torch.arange(T, device=device).unsqueeze(0).expand(B, -1)
        cos, sin = self.rotary_emb(x, position_ids)
        q, k = apply_rotary_pos_emb(q, k, cos, sin)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        causal_mask = torch.triu(torch.ones(T, T, device=device, dtype=torch.bool), diagonal=1)
        att = att.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        att = F.softmax(att, dim=-1, dtype=torch.float32).to(x.dtype)

        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(out)

In [ ]:
class SwiGLUFFN(nn.Module):
    def __init__(self, d_model: int, intermediate_size: int, linear_cls):
        super().__init__()
        self.gate_proj = linear_cls(d_model, intermediate_size, bias=False)
        self.up_proj   = linear_cls(d_model, intermediate_size, bias=False)
        self.down_proj = linear_cls(intermediate_size, d_model, bias=False)

    def forward(self, x: Tensor) -> Tensor:
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


In [ ]:
class Block(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int,
                 intermediate_size: int, rope_base: float, linear_cls):
        super().__init__()
        self.input_layernorm = RMSNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, n_heads, max_seq_len, rope_base, linear_cls)
        self.post_attention_layernorm = RMSNorm(d_model)
        self.mlp = SwiGLUFFN(d_model, intermediate_size, linear_cls)

    def forward(self, x: Tensor) -> Tensor:
        x = x + self.self_attn(self.input_layernorm(x))
        x = x + self.mlp(self.post_attention_layernorm(x))
        return x

In [ ]:
class LLaMAModel(nn.Module):
    """
    Unified model. Pass linear_cls=BitLinear for BitNet, linear_cls=nn.Linear for baseline.
    Everything else is identical.
    """
    def __init__(self, vocab_size: int, block_size: int, d_model: int = 256,
                 n_layers: int = 4, n_heads: int = 4, intermediate_size: int = None,
                 rope_base: float = 10000.0, tie_word_embeddings: bool = True,
                 linear_cls=nn.Linear):
        super().__init__()
        self.block_size = block_size
        self.d_model = d_model
        self.tie_word_embeddings = tie_word_embeddings
        self.model_type = "BitNet b1.58" if linear_cls is BitLinear else "FP32 Baseline"

        if intermediate_size is None:
            intermediate_size = compute_intermediate_size(d_model)

        self.embed_tokens = nn.Embedding(vocab_size, d_model)

        self.layers = nn.ModuleList([
            Block(d_model, n_heads, block_size, intermediate_size, rope_base, linear_cls)
            for _ in range(n_layers)
        ])

        self.norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        if tie_word_embeddings:
            self.lm_head.weight = self.embed_tokens.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx: Tensor, targets: Tensor = None) -> tuple:
        B, T = idx.shape
        x = self.embed_tokens(idx)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx: Tensor, max_new_tokens: int, temperature: float = 1.0,
                 top_k: int = None) -> Tensor:
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

    def count_parameters(self) -> dict:
        total = sum(p.numel() for p in self.parameters())
        embedding = sum(p.numel() for p in self.embed_tokens.parameters())
        lm_head_params = 0 if self.tie_word_embeddings else sum(p.numel() for p in self.lm_head.parameters())
        norm_params = sum(p.numel() for n, p in self.named_parameters() if 'norm' in n or 'layernorm' in n)
        linear_params = total - embedding - lm_head_params - norm_params
        return {
            "total": total,
            "embedding": embedding,
            "lm_head (tied)": lm_head_params,
            "norm": norm_params,
            "linear (core)": linear_params,
        }


In [ ]:
def prepare_data(block_size: int = 128, max_train_tokens: int = 10_000_000,
                 max_val_tokens: int = 1_000_000):
    model_name = "gpt2"
    num_proc = 4

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    vocab_size = tokenizer.vocab_size
    print(f"Vocab size: {vocab_size}")

    print("Loading Wikitext-103...")
    raw_datasets = load_dataset("wikitext", "wikitext-103-raw-v1")
    raw_datasets = raw_datasets.filter(
        lambda x: x["text"] is not None and x["text"].strip() != ""
    )

    tokenizer.model_max_length = int(1e9)
    def tokenize_function(examples):
        return tokenizer(examples["text"])

    tokenized_datasets = raw_datasets.map(
        tokenize_function, batched=True, num_proc=num_proc, remove_columns=["text"]
    )

    def group_texts(examples):
        concatenated = {k: list(chain.from_iterable(examples[k])) for k in examples.keys()}
        total_length = (len(concatenated["input_ids"]) // block_size) * block_size
        result = {
            k: [t[i:i + block_size] for i in range(0, total_length, block_size)]
            for k, t in concatenated.items()
        }
        result["labels"] = result["input_ids"].copy()
        return result

    lm_datasets = tokenized_datasets.map(group_texts, batched=True, num_proc=num_proc)

    train_tokens = list(chain.from_iterable(lm_datasets["train"]["input_ids"]))
    val_tokens = list(chain.from_iterable(lm_datasets["validation"]["input_ids"]))

    train_tokens = train_tokens[:max_train_tokens]
    val_tokens = val_tokens[:max_val_tokens]

    train_data = torch.tensor(train_tokens, dtype=torch.long)
    val_data = torch.tensor(val_tokens, dtype=torch.long)
    print(f"Train tokens: {len(train_data):,}, Val tokens: {len(val_data):,}")
    return train_data, val_data, vocab_size, tokenizer


def get_batch(data: Tensor, batch_size: int, block_size: int, device: torch.device):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    xb = torch.stack([data[i:i + block_size] for i in ix]).to(device)
    yb = torch.stack([data[i + 1:i + block_size + 1] for i in ix]).to(device)
    return xb, yb

In [ ]:
def get_lr(step: int, max_steps: int, max_lr: float, warmup_steps: int = 750,
           min_lr_ratio: float = 0.1) -> float:
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    if step >= max_steps:
        return max_lr * min_lr_ratio
    decay_ratio = (step - warmup_steps) / (max_steps - warmup_steps)
    return max_lr * (1.0 - decay_ratio * (1.0 - min_lr_ratio))

In [ ]:
def train_model(model, train_data, val_data, device, config):
    model.to(device)
    print(f"\n{'='*70}")
    print(f"Training: {model.model_type}")
    print(f"{'='*70}")
    param_info = model.count_parameters()
    for k, v in param_info.items():
        print(f"  {k}: {v:,}")

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config['max_lr'],
        betas=(0.9, 0.98), weight_decay=0.01,
    )

    steps_log, train_losses, train_accs, train_ppls = [], [], [], []
    val_losses, val_accs, val_ppls = [], [], []
    best_val_loss = float('inf')
    start_time = time.time()

    for step in range(config['max_steps'] + 1):
        lr = get_lr(step, config['max_steps'], config['max_lr'], config['warmup_steps'])
        for pg in optimizer.param_groups:
            pg['lr'] = lr

        model.train()
        xb, yb = get_batch(train_data, config['batch_size'], config['block_size'], device)
        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        if step % config['eval_interval'] == 0:
            model.eval()
            train_loss = loss.item()
            train_acc = (logits.argmax(-1) == yb).float().mean().item()
            train_ppl = math.exp(min(train_loss, 20))

            val_loss_sum, val_acc_sum = 0.0, 0.0
            with torch.no_grad():
                for _ in range(config['n_val_batches']):
                    xv, yv = get_batch(val_data, config['batch_size'], config['block_size'], device)
                    logits_v, loss_v = model(xv, yv)
                    val_loss_sum += loss_v.item()
                    val_acc_sum += (logits_v.argmax(-1) == yv).float().mean().item()

            avg_val_loss = val_loss_sum / config['n_val_batches']
            avg_val_acc = val_acc_sum / config['n_val_batches']
            avg_val_ppl = math.exp(min(avg_val_loss, 20))

            steps_log.append(step)
            train_losses.append(train_loss)
            train_accs.append(train_acc)
            train_ppls.append(train_ppl)
            val_losses.append(avg_val_loss)
            val_accs.append(avg_val_acc)
            val_ppls.append(avg_val_ppl)

            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss

            elapsed = time.time() - start_time
            print(f"[{model.model_type}] Step {step:5d} | LR {lr:.2e} | "
                  f"Train {train_loss:.4f}/{train_acc:.4f}/{train_ppl:.1f} | "
                  f"Val {avg_val_loss:.4f}/{avg_val_acc:.4f}/{avg_val_ppl:.1f} | "
                  f"{elapsed:.0f}s")

    total_time = time.time() - start_time
    print(f"\n{model.model_type} finished in {total_time:.1f}s | Best val loss: {best_val_loss:.4f}")

    return {
        'steps': steps_log,
        'train_loss': train_losses, 'train_acc': train_accs, 'train_ppl': train_ppls,
        'val_loss': val_losses, 'val_acc': val_accs, 'val_ppl': val_ppls,
        'best_val_loss': best_val_loss, 'total_time': total_time,
    }

In [ ]:
def plot_comparison(bitnet_metrics, baseline_metrics, filename='comparison.png'):
    fig, axes = plt.subplots(3, 2, figsize=(16, 14))
    fig.suptitle('BitNet b1.58 vs FP32 Baseline — Fair Comparison\n'
                 '(Identical architecture, only difference: BitLinear vs nn.Linear)',
                 fontsize=14, fontweight='bold')

    metrics = [
        ('train_loss', 'val_loss', 'Loss', 'Loss'),
        ('train_acc', 'val_acc', 'Accuracy', 'Accuracy'),
        ('train_ppl', 'val_ppl', 'Perplexity', 'Perplexity'),
    ]

    for row, (train_key, val_key, ylabel, title) in enumerate(metrics):
        ax = axes[row, 0]
        ax.plot(bitnet_metrics['steps'], bitnet_metrics[train_key],
                label='BitNet b1.58', color='#e74c3c', linewidth=1.5)
        ax.plot(baseline_metrics['steps'], baseline_metrics[train_key],
                label='FP32 Baseline', color='#3498db', linewidth=1.5)
        ax.set_xlabel('Step'); ax.set_ylabel(ylabel)
        ax.set_title(f'Train {title}'); ax.legend(); ax.grid(True, alpha=0.3)

        ax = axes[row, 1]
        ax.plot(bitnet_metrics['steps'], bitnet_metrics[val_key],
                label='BitNet b1.58', color='#e74c3c', linewidth=1.5)
        ax.plot(baseline_metrics['steps'], baseline_metrics[val_key],
                label='FP32 Baseline', color='#3498db', linewidth=1.5)
        ax.set_xlabel('Step'); ax.set_ylabel(ylabel)
        ax.set_title(f'Validation {title}'); ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Plot saved as '{filename}'")

In [ ]:
def print_final_summary(bitnet_metrics, baseline_metrics, bitnet_model, baseline_model):
    print("\n" + "="*70)
    print("FINAL COMPARISON SUMMARY")
    print("="*70)
    print(f"{'Metric':<30} {'BitNet b1.58':>15} {'FP32 Baseline':>15} {'Δ':>10}")
    print("-"*70)

    bn, bl = bitnet_metrics, baseline_metrics
    rows = [
        ("Best Val Loss",     bn['best_val_loss'],  bl['best_val_loss'],  bn['best_val_loss'] - bl['best_val_loss']),
        ("Final Val Loss",    bn['val_loss'][-1],    bl['val_loss'][-1],    bn['val_loss'][-1] - bl['val_loss'][-1]),
        ("Final Val Acc",     bn['val_acc'][-1],     bl['val_acc'][-1],     bn['val_acc'][-1] - bl['val_acc'][-1]),
        ("Final Val PPL",     bn['val_ppl'][-1],     bl['val_ppl'][-1],     bn['val_ppl'][-1] - bl['val_ppl'][-1]),
        ("Training Time (s)", bn['total_time'],      bl['total_time'],      bn['total_time'] - bl['total_time']),
    ]
    for name, bv, blv, delta in rows:
        print(f"{name:<30} {bv:>15.4f} {blv:>15.4f} {delta:>+10.4f}")

    bn_params = bitnet_model.count_parameters()
    bl_params = baseline_model.count_parameters()
    print(f"\n{'Parameters (total)':<30} {bn_params['total']:>15,} {bl_params['total']:>15,}")
    print(f"{'Parameters (linear core)':<30} {bn_params['linear (core)']:>15,} {bl_params['linear (core)']:>15,}")

    bitlinear_params = bn_params['linear (core)']
    fp_params = bn_params['total'] - bitlinear_params
    effective_bits = (bitlinear_params * 1.58 + fp_params * 32) / bn_params['total']
    print(f"\n{'BitNet effective bits/param':<30} {effective_bits:>15.2f}")
    print(f"{'BitNet memory ratio vs FP32':<30} {effective_bits/32:>15.2%}")
    print("="*70)

In [ ]:
# ── Shared config ──
config = {
    'block_size': 128,
    'd_model': 256,
    'n_layers': 4,
    'n_heads': 4,
    'batch_size': 32,
    'max_lr': 1e-3,
    'max_steps': 10000,
    'eval_interval': 200,
    'n_val_batches': 20,
    'warmup_steps': 750,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ── Data (loaded ONCE, shared by both models) ──
train_data, val_data, vocab_size, tokenizer = prepare_data(block_size=config['block_size'])
config['block_size'] = min(config['block_size'], len(train_data) - 1)

intermediate_size = compute_intermediate_size(config['d_model'])
print(f"Intermediate size (SwiGLU): {intermediate_size}")

# ── Build both models with IDENTICAL config + same random seed ──
model_kwargs = dict(
    vocab_size=vocab_size,
    block_size=config['block_size'],
    d_model=config['d_model'],
    n_layers=config['n_layers'],
    n_heads=config['n_heads'],
    intermediate_size=intermediate_size,
    rope_base=10000.0,
    tie_word_embeddings=True,
)

torch.manual_seed(42)
baseline_model = LLaMAModel(**model_kwargs, linear_cls=nn.Linear)

torch.manual_seed(42)
bitnet_model = LLaMAModel(**model_kwargs, linear_cls=BitLinear)

bn_p = bitnet_model.count_parameters()['total']
bl_p = baseline_model.count_parameters()['total']
print(f"\nBitNet params: {bn_p:,} | Baseline params: {bl_p:,} | Match: {bn_p == bl_p}")

# ── Train both sequentially ──
baseline_metrics = train_model(baseline_model, train_data, val_data, device, config)
bitnet_metrics   = train_model(bitnet_model,   train_data, val_data, device, config)

# ── Compare ──
plot_comparison(bitnet_metrics, baseline_metrics)
print_final_summary(bitnet_metrics, baseline_metrics, bitnet_model, baseline_model)

# ── Generate from both ──
for model in [baseline_model, bitnet_model]:
    model.eval()
    prompt = "The meaning of life is"
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated = model.generate(input_ids, max_new_tokens=50, temperature=0.8, top_k=50)
    print(f"\n[{model.model_type}] {tokenizer.decode(generated[0])}")

Device: cuda
Vocab size: 50257
Loading Wikitext-103...


Filter:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3760 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/2891 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/1165029 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/2461 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/2891 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/1165029 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/2461 [00:00<?, ? examples/s]

Train tokens: 10,000,000, Val tokens: 247,040
Intermediate size (SwiGLU): 768

BitNet params: 16,275,968 | Baseline params: 16,275,968 | Match: True

Training: FP32 Baseline
  total: 16,275,968
  embedding: 12,865,792
  lm_head (tied): 0
  norm: 2,304
  linear (core): 3,407,872
[FP32 Baseline] Step     0 | LR 1.33e-06 | Train 10.8713/0.0000/52644.9 | Val 10.8744/0.0002/52805.0 | 2s
[FP32 Baseline] Step   200 | LR 2.68e-04 | Train 7.1044/0.1392/1217.3 | Val 7.0365/0.1526/1137.4 | 32s
[FP32 Baseline] Step   400 | LR 5.35e-04 | Train 6.2206/0.1919/503.0 | Val 6.1816/0.2031/483.8 | 64s
[FP32 Baseline] Step   600 | LR 8.01e-04 | Train 5.7228/0.2207/305.8 | Val 5.8442/0.2130/345.2 | 97s
[FP32 Baseline] Step   800 | LR 9.95e-04 | Train 5.4507/0.2344/232.9 | Val 5.6048/0.2260/271.7 | 133s
[FP32 Baseline] Step  1000 | LR 9.76e-04 | Train 5.3097/0.2354/202.3 | Val 5.4188/0.2316/225.6 | 168s
[FP32 Baseline] Step  1200 | LR 9.56e-04 | Train 5.5071/0.2107/246.4 | Val 5.2677/0.2444/194.0 | 204s
[FP3